# 05.3 — Feature Forensik Analizi
Her bir özelliğin **tek başına Rank IC** değerini ölçer.
Gürültü yapan featureları tespit edip, modelden kesmek için kullanılır.

**Çıktı:** Feature bazlı IC tablosu + kategori bazlı performans + eleme önerileri.

In [ ]:
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml tqdm pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json

REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone https://github.com/umutergul74/Scalp2.git {REPO_DIR}

sys.path.insert(0, REPO_DIR)

import logging
logging.basicConfig(level=logging.WARNING)

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

from scalp2.config import load_config
config = load_config(f'{REPO_DIR}/config.yaml')
DATA_DIR = '/content/drive/MyDrive/scalp2/data/processed'

print('Setup complete.')

In [ ]:
# Load labeled dataset and feature list
df = pd.read_parquet(f'{DATA_DIR}/BTC_USDT_labeled.parquet')
with open(f'{DATA_DIR}/feature_columns.json', 'r') as f:
    feature_cols = json.load(f)

# Forward returns
close = df['close'].values
fwd_ret_1 = np.full(len(close), np.nan)
fwd_ret_1[:-1] = (close[1:] - close[:-1]) / close[:-1]

fwd_ret_10 = np.full(len(close), np.nan)
fwd_ret_10[:-10] = (close[10:] - close[:-10]) / close[:-10]

print(f'Dataset: {len(df):,} rows, {len(feature_cols)} features')
print(f'Feature list sample: {feature_cols[:10]}')

In [ ]:
# ============================================================
#  PER-FEATURE RANK IC COMPUTATION
#  Each feature's raw Spearman correlation with future returns
# ============================================================

# Use a representative sample (middle 60% of data to avoid warmup/edge effects)
n = len(df)
start_idx = int(n * 0.2)
end_idx = int(n * 0.8)

results = []

for col in feature_cols:
    vals = df[col].values[start_idx:end_idx]
    r1 = fwd_ret_1[start_idx:end_idx]
    r10 = fwd_ret_10[start_idx:end_idx]
    
    # Skip if all NaN or constant
    valid1 = ~np.isnan(vals) & ~np.isnan(r1)
    valid10 = ~np.isnan(vals) & ~np.isnan(r10)
    
    if valid1.sum() < 100 or valid10.sum() < 100:
        results.append({'feature': col, 'ic_1bar': np.nan, 'ic_10bar': np.nan, 'std': np.nan})
        continue
    
    ic1 = spearmanr(vals[valid1], r1[valid1]).statistic
    ic10 = spearmanr(vals[valid10], r10[valid10]).statistic
    feat_std = np.nanstd(vals)
    
    results.append({
        'feature': col,
        'ic_1bar': ic1,
        'ic_10bar': ic10,
        'std': feat_std,
    })

ic_df = pd.DataFrame(results)
ic_df['abs_ic_10'] = ic_df['ic_10bar'].abs()
ic_df = ic_df.sort_values('abs_ic_10', ascending=False)

print(f'\nAnalyzed {len(ic_df)} features')
print(f'Features with |IC(10)| > 0.02: {(ic_df["abs_ic_10"] > 0.02).sum()}')
print(f'Features with |IC(10)| < 0.005: {(ic_df["abs_ic_10"] < 0.005).sum()} (noise candidates)')

In [ ]:
# ============================================================
#  TOP 30 BEST FEATURES (Highest Absolute IC)
# ============================================================
print('=' * 70)
print('  TOP 30 FEATURES — En Yüksek Tahmin Gücü (|IC(10-bar)|)')
print('=' * 70)
print(f'{"Feature":<40} {"IC(1-bar)":>10} {"IC(10-bar)":>10} {"Std":>10}')
print('-' * 70)
for _, row in ic_df.head(30).iterrows():
    print(f'{row["feature"]:<40} {row["ic_1bar"]:>+10.4f} {row["ic_10bar"]:>+10.4f} {row["std"]:>10.4f}')

In [ ]:
# ============================================================
#  BOTTOM 30 WORST FEATURES (Noise Candidates)
# ============================================================
print('=' * 70)
print('  BOTTOM 30 FEATURES — Gürültü Adayları (|IC(10-bar)| en düşük)')
print('=' * 70)
print(f'{"Feature":<40} {"IC(1-bar)":>10} {"IC(10-bar)":>10} {"Std":>10}')
print('-' * 70)
for _, row in ic_df.tail(30).iterrows():
    print(f'{row["feature"]:<40} {row["ic_1bar"]:>+10.4f} {row["ic_10bar"]:>+10.4f} {row["std"]:>10.4f}')

In [ ]:
# ============================================================
#  CATEGORY-LEVEL ANALYSIS
# ============================================================

def categorize_feature(name):
    # Strip MTF prefix
    base = name
    for pfx in ('1h_', '4h_'):
        if name.startswith(pfx):
            base = name[len(pfx):]
            break
    
    if any(k in base for k in ['cvd', 'taker', 'absorb', 'vpin', 'kyle', 'amihud']):
        return 'Microstructure'
    elif any(k in base for k in ['vol_per_trade', 'vpt', 'high_activity', 'whale']):
        return 'Whale Detector'
    elif any(k in base for k in ['rsi', 'macd', 'stoch', 'adx', 'plus_di', 'minus_di', 'bb_']):
        return 'Technical Oscillators'
    elif any(k in base for k in ['ema_', 'log_return', 'volume_ratio', 'volume_zscore']):
        return 'Trend/Volume'
    elif any(k in base for k in ['gk_vol', 'park_vol', 'vol_ratio', 'vol_zscore', 'atr_pct']):
        return 'Volatility'
    elif any(k in base for k in ['wavelet', 'denoised', 'noise']):
        return 'Wavelet'
    elif any(k in base for k in ['fvg', 'swing', 'sweep', 'bos', 'choch', 'ob_', 'vwap', 'smart']):
        return 'Smart Money'
    elif any(k in base for k in ['funding', 'oi_']):
        return 'Derivatives'
    else:
        return 'Other'

ic_df['category'] = ic_df['feature'].apply(categorize_feature)

cat_summary = ic_df.groupby('category').agg(
    count=('feature', 'count'),
    mean_ic1=('ic_1bar', 'mean'),
    mean_ic10=('ic_10bar', 'mean'),
    mean_abs_ic10=('abs_ic_10', 'mean'),
    max_abs_ic10=('abs_ic_10', 'max'),
).sort_values('mean_abs_ic10', ascending=False)

print('=' * 80)
print('  KATEGORİ BAZINDA PERFORMANS')
print('=' * 80)
print(f'{"Category":<25} {"Count":>6} {"Mean IC(1)":>10} {"Mean IC(10)":>12} {"Mean|IC(10)|":>13} {"Max|IC(10)|":>12}')
print('-' * 80)
for cat, row in cat_summary.iterrows():
    print(f'{cat:<25} {row["count"]:>6.0f} {row["mean_ic1"]:>+10.4f} {row["mean_ic10"]:>+12.4f} {row["mean_abs_ic10"]:>13.4f} {row["max_abs_ic10"]:>12.4f}')

In [ ]:
# ============================================================
#  MICROSTRUCTURE FEATURES DEEP DIVE
#  (The new God Mode features — are they helping or hurting?)
# ============================================================
micro_df = ic_df[ic_df['category'] == 'Microstructure'].sort_values('abs_ic_10', ascending=False)

print('=' * 70)
print('  MICROSTRUCTURE FEATURES — Detaylı IC Analizi')
print('=' * 70)
print(f'{"Feature":<40} {"IC(1-bar)":>10} {"IC(10-bar)":>10} {"Verdict":>10}')
print('-' * 70)
for _, row in micro_df.iterrows():
    verdict = 'ALFA' if row['abs_ic_10'] > 0.01 else ('ZAYIF' if row['abs_ic_10'] > 0.005 else 'GÜRÜLTÜ')
    emoji = '🟢' if verdict == 'ALFA' else ('🟡' if verdict == 'ZAYIF' else '🔴')
    print(f'{row["feature"]:<40} {row["ic_1bar"]:>+10.4f} {row["ic_10bar"]:>+10.4f} {emoji} {verdict}')

# Whale detector
whale_df = ic_df[ic_df['category'] == 'Whale Detector'].sort_values('abs_ic_10', ascending=False)
if len(whale_df) > 0:
    print('\n' + '=' * 70)
    print('  WHALE DETECTOR FEATURES')
    print('=' * 70)
    for _, row in whale_df.iterrows():
        verdict = 'ALFA' if row['abs_ic_10'] > 0.01 else ('ZAYIF' if row['abs_ic_10'] > 0.005 else 'GÜRÜLTÜ')
        emoji = '🟢' if verdict == 'ALFA' else ('🟡' if verdict == 'ZAYIF' else '🔴')
        print(f'{row["feature"]:<40} {row["ic_1bar"]:>+10.4f} {row["ic_10bar"]:>+10.4f} {emoji} {verdict}')

In [ ]:
# ============================================================
#  VISUALIZATION — Feature IC Heatmap
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 12))

# Top 40 by |IC(10)|
top40 = ic_df.head(40)
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in top40['ic_10bar']]
axes[0].barh(range(len(top40)), top40['ic_10bar'].values, color=colors)
axes[0].set_yticks(range(len(top40)))
axes[0].set_yticklabels(top40['feature'].values, fontsize=8)
axes[0].set_xlabel('Rank IC (10-bar)')
axes[0].set_title('TOP 40 Features — IC(10-bar)', fontweight='bold')
axes[0].invert_yaxis()
axes[0].axvline(x=0, color='black', linewidth=0.5)

# Category comparison
cat_data = cat_summary.sort_values('mean_abs_ic10', ascending=True)
colors2 = plt.cm.viridis(np.linspace(0.2, 0.9, len(cat_data)))
axes[1].barh(range(len(cat_data)), cat_data['mean_abs_ic10'].values, color=colors2)
axes[1].set_yticks(range(len(cat_data)))
axes[1].set_yticklabels([f"{c} ({int(cat_data.loc[c, 'count'])} feat)" for c in cat_data.index], fontsize=9)
axes[1].set_xlabel('Mean |IC(10-bar)|')
axes[1].set_title('Category Performance — Ortalama |IC|', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/scalp2/feature_forensics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to /content/drive/MyDrive/scalp2/feature_forensics.png')

In [ ]:
# ============================================================
#  FINAL VERDICT — Noise features to consider removing
# ============================================================
noise_threshold = 0.005  # Features with |IC(10)| < 0.005 are noise
noise_features = ic_df[ic_df['abs_ic_10'] < noise_threshold]['feature'].tolist()

weak_threshold = 0.01
weak_features = ic_df[(ic_df['abs_ic_10'] >= noise_threshold) & (ic_df['abs_ic_10'] < weak_threshold)]['feature'].tolist()

strong_features = ic_df[ic_df['abs_ic_10'] >= 0.02]['feature'].tolist()

print('=' * 70)
print('  SON KARAR — Feature Eleme Raporu')
print('=' * 70)
print(f'\n🟢 GÜÇLÜ (|IC| >= 0.02): {len(strong_features)} feature')
for f in strong_features:
    row = ic_df[ic_df['feature'] == f].iloc[0]
    print(f'   {f:<40} IC(10)={row["ic_10bar"]:>+.4f}')

print(f'\n🟡 ZAYIF (0.005 <= |IC| < 0.01): {len(weak_features)} feature')
for f in weak_features[:15]:
    row = ic_df[ic_df['feature'] == f].iloc[0]
    print(f'   {f:<40} IC(10)={row["ic_10bar"]:>+.4f}')
if len(weak_features) > 15:
    print(f'   ... ve {len(weak_features) - 15} tane daha')

print(f'\n🔴 GÜRÜLTÜ (|IC| < 0.005): {len(noise_features)} feature — ELENMELİ!')
for f in noise_features:
    row = ic_df[ic_df['feature'] == f].iloc[0]
    print(f'   {f:<40} IC(10)={row["ic_10bar"]:>+.4f}')

print(f'\n\nToplam: {len(feature_cols)} feature')
print(f'Eleme sonrası kalacak: {len(feature_cols) - len(noise_features)} feature')
print(f'Boyut azaltma: %{100 * len(noise_features) / len(feature_cols):.1f}')

In [ ]:
# ============================================================
#  SAVE — Export noise list for automated pruning
# ============================================================
report = {
    'noise_features': noise_features,
    'weak_features': weak_features,
    'strong_features': strong_features,
    'all_ic': ic_df[['feature', 'ic_1bar', 'ic_10bar', 'category']].to_dict('records'),
}

import json
report_path = f'{DATA_DIR}/feature_forensics_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Report saved to {report_path}')
print('\nBu raporu bana (AI\'a) kopyalayıp yapıştırın — gürültü featurelarını otomatik eleyeceğim.')